$$\hat{y} = \arg\max_{c} P(c \mid x)$$

$$P(c \mid x) = \frac{P(x \mid c)\,P(c)}{P(x)}$$

$$\hat{y} = \arg\max_{c} P(x \mid c)\,P(c)$$

$$P(x \mid c) = P(x_1, \ldots, x_V \mid c) = \prod_{i=1}^{V} P(x_i \mid c)$$

$$\hat{y} = \arg\max_{c} P(c) \prod_{i=1}^{V} P(x_i \mid c)$$

$$\hat{y} = \arg\max_{c} \left[ \log P(c) + \sum_{i=1}^{V} \log P(x_i \mid c) \right]$$

### Estimating the prior P(c) by MLE

Treat class labels as draws from a categorical distribution with parameters $\theta_c = P(c)$, constrained by $\sum_c \theta_c = 1$. With $N_c$ samples in class $c$ out of $N$, the log-likelihood of the labels is:

$$\ell(\theta) = \sum_c N_c \log \theta_c$$

Maximize subject to $\sum_c \theta_c = 1$ using a Lagrange multiplier:

$$\mathcal{L} = \sum_c N_c \log \theta_c + \lambda\left(1 - \sum_c \theta_c\right)$$

$$\frac{\partial \mathcal{L}}{\partial \theta_c} = \frac{N_c}{\theta_c} - \lambda = 0 \;\Rightarrow\; \theta_c = \frac{N_c}{\lambda}$$

Apply the constraint $\sum_c \theta_c = 1 \Rightarrow \sum_c N_c/\lambda = N/\lambda = 1 \Rightarrow \lambda = N$. Therefore:

$$\boxed{P(c) = \frac{N_c}{N}}$$

### Estimating $P(x_i \mid c)$ — the multinomial likelihood

The parameter $\phi_c = (\phi_{c1}, \dots, \phi_{cV})$ is a probability vector: $\phi_{ci} \ge 0$ and $\sum_i \phi_{ci} = 1$. The natural prior over such a vector is the Dirichlet distribution. Its general form with concentration parameters $(a_1, \dots, a_V)$ is:

$$P(\phi_c) = \frac{1}{B(a)} \prod_{i=1}^V \phi_{ci}^{\,a_i - 1}$$

where $B(a)$ is the normalizing constant (the multivariate Beta function). We choose a symmetric Dirichlet — same concentration on every word, since a priori no vocabulary word is special — by setting every $a_i$ equal. To get add-$\alpha$ smoothing we set:

$$a_i = \alpha + 1 \quad \text{for all } i$$

Substituting $a_i - 1 = \alpha$:

$$P(\phi_c) = \frac{1}{B(\alpha+1,\dots,\alpha+1)} \prod_{i=1}^V \phi_{ci}^{\,\alpha}$$

The normalizing constant $B(\cdot)$ does not depend on $\phi_c$ — it's a fixed number once $\alpha$ and $V$ are chosen. So as a function of $\phi_c$:

$$\boxed{P(\phi_c) \propto \prod_{i=1}^V \phi_{ci}^{\,\alpha}}$$

The multinomial likelihood of the class-$c$ counts $T_{ci}$ (dropping the multinomial coefficient, which is constant in $\phi_c$):

$$\boxed{P(\text{data} \mid \phi_c) \propto \prod_{i=1}^V \phi_{ci}^{\,T_{ci}}}$$

$$P(\phi_c \mid \text{data}) \;\propto\; \prod_{i=1}^V \phi_{ci}^{\,T_{ci}} \cdot \prod_{i=1}^V \phi_{ci}^{\,\alpha} = \prod_{i=1}^V \phi_{ci}^{\,T_{ci}} \,\phi_{ci}^{\,\alpha} = \prod_{i=1}^V \phi_{ci}^{\,T_{ci} + \alpha}$$

$$\log P(\phi_c \mid \text{data}) = \underbrace{\sum_i T_{ci} \log \phi_{ci}}_{\text{log-likelihood}} + \underbrace{\sum_i \alpha \log \phi_{ci}}_{\text{log-prior}} + \text{const}$$

$$\boxed{P(x_i \mid c) = \frac{T_{ci} + \alpha}{\sum_j T_{cj} + \alpha V}}$$

## Final scoring formula

Putting the estimates into the log decision rule. For a test document with counts $x_i$:

$$\log P(c \mid x) \;\overset{+}{=}\; \underbrace{\log \frac{N_c}{N}}_{\text{prior}} + \sum_{i=1}^V x_i \underbrace{\log \frac{T_{ci} + \alpha}{\sum_j T_{cj} + \alpha V}}_{\log \phi_{ci}}$$

(The $x_i$ multiplier appears because a word occurring $x_i$ times contributes $\log \phi_{ci}$ that many times: $\log \phi_{ci}^{x_i} = x_i \log \phi_{ci}$.) This is exactly the `X @ log_likelihood.T + log_prior` matmul in the code.